<div style="display: flex; background-color: RGB(128,0,32);">
    <h1 style="margin: auto; padding: 30px; color: #fff;">P13 — SEGMENTATION DU CATALOGUE BOTTLENECK PAR CLUSTERING</h1>
</div>

# OBJECTIF DE CE NOTEBOOK

Ce notebook prolonge le **projet 6** (analyse du stock et des ventes du site BottleNeck).
Là où le P6 décrivait le catalogue **variable par variable** (analyse univariée du prix, du CA,
des quantités, de la marge), l'objectif ici est de **croiser ces dimensions** pour faire émerger,
de façon automatisée et data-driven, des **segments de produits** exploitables par le métier.

L'amélioration apportée s'appuie sur une méthode de **machine learning non supervisé (clustering)**,
testée et justifiée de manière critique : on compare plusieurs algorithmes et on documente les choix.

<div style="background-color: RGB(0,77,64);">
    <h2 style="margin: auto; padding: 20px; color: #fff;">Démarche</h2>
</div>

**1. Pourquoi un notebook séparé ?**
Le P6 reste le notebook de *nettoyage et d'exploration*. Ce notebook repart directement du
**jeu de données consolidé et nettoyé** exporté à l'issue du P6 (`df_merge`). On évite ainsi de
ré-exécuter toute la chaîne de préparation, on gagne en reproductibilité et on garde un livrable lisible.

**2. Pourquoi le format Parquet ?**
L'export est stocké en **`.parquet`** (et non `.xlsx` ou `.pkl`) : il **préserve les types de données**,
reste **compact, rapide et portable**, et constitue le format standard de l'écosystème data moderne —
contrairement au pickle, opaque et fragile aux changements de version.

**3. La problématique métier**
> *« Tous les produits du catalogue ne se valent pas. Peut-on les regrouper automatiquement en
> familles cohérentes (positionnement, volume de ventes, rentabilité) pour orienter les décisions
> d'assortiment, de pricing et de gestion de stock ? »*

On s'inspire de la logique d'une **matrice de portefeuille** (type BCG), mais en version
**multivariée et automatisée** : ce n'est pas l'analyste qui décide des segments à l'avance,
c'est l'algorithme qui les révèle à partir des données, l'analyste se chargeant de les **interpréter et nommer**.

<div style="background-color: RGB(0,77,64);">
    <h2 style="margin: auto; padding: 20px; color: #fff;">Variables retenues pour la segmentation</h2>
</div>

| Variable | Dimension métier | Traitement |
|---|---|---|
| `price` | Positionnement (premium ↔ entrée de gamme) | log + standardisation |
| `total_sales` | Volume de ventes | log + standardisation |
| `tx_marge` | Rentabilité | standardisation |

> ⚠️ La variable `ca` est **volontairement exclue** du clustering : étant le produit
> `total_sales × price`, elle est colinéaire aux variables d'entrée et fausserait la segmentation.
> Elle reste utilisée *en lecture* pour qualifier les segments a posteriori.

<div style="background-color: RGB(0,77,64);">
    <h2 style="margin: auto; padding: 20px; color: #fff;">Approche comparative</h2>
</div>

- **Choix du nombre de segments (k)** justifié par le **score de silhouette** (et méthode du coude).
- **Comparaison de deux algorithmes** : `KMeans` vs **clustering hiérarchique** (`AgglomerativeClustering`),
  pour valider la robustesse de la segmentation obtenue.
- Toutes les variables asymétriques sont transformées (`log1p`) et **standardisées** avant clustering.

# EDA

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

from tools import sherlock

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score


df = pd.read_parquet(r"..\data\df_merge.parquet")
sherlock(df)

714 lignes | 16 colonnes | 0 lignes doublons
--------------------------------------------------
                          Type  Manquants Manquants %  Uniques
product_id               int64          0        0.0%      714
onsale_web               int64          0        0.0%        2
price                  float64          0        0.0%      362
stock_quantity           int64          0        0.0%       81
stock_status            object          0        0.0%        2
purchase_price         float64          0        0.0%      590
total_sales            float64          0        0.0%       24
post_date       datetime64[us]          0        0.0%      714
product_type            object          1       0.14%        6
post_title              object          0        0.0%      711
price_zscore           float64          0        0.0%      362
ca                     float64          0        0.0%      531
stock_mois             float64          0        0.0%      166
val_stock             

In [2]:
df['tx_marge'].describe()

count    714.000000
mean      61.017018
std        9.625244
min      -86.394338
25%       56.578632
50%       61.269842
75%       66.293423
max       91.412471
Name: tx_marge, dtype: float64

In [3]:
mask_invendus = df['total_sales'] == 0
df_final = df[~mask_invendus]

In [4]:
anciennete = (df_final['post_date'].max() - df_final['post_date']).dt.days
anciennete.describe()

count    689.000000
mean     695.142235
std      216.869100
min        0.000000
25%      531.000000
50%      808.000000
75%      872.000000
max      892.000000
Name: post_date, dtype: float64

In [ ]:
df_final['total_sales'].describe()

count    689.000000
mean       8.346880
std        3.937391
min        1.000000
25%        5.000000
50%        8.000000
75%       11.000000
max       36.000000
Name: total_sales, dtype: float64

In [6]:
np.log1p(df_final['total_sales']).describe()

count    689.000000
mean       2.137743
std        0.465335
min        0.693147
25%        1.791759
50%        2.197225
75%        2.484907
max        3.610918
Name: total_sales, dtype: float64

# ML

In [7]:
(df_final['price'] <= 0).sum()

np.int64(0)

### Cellule 1 — feature engineering

In [8]:
df_final = df_final.copy()

# Ancienneté ancrée sur une date FIXE (repro) — pas de datetime.today()
df_final['anciennete'] = (df_final['post_date'].max() - df_final['post_date']).dt.days

# Axes : price & sales en log (asymétrie droite), ancienneté brute (asymétrie gauche)
feat = pd.DataFrame(index=df_final.index)
feat['price_log']  = np.log1p(df_final['price'])
feat['sales_log']  = np.log1p(df_final['total_sales'])
feat['anciennete'] = df_final['anciennete']

# Standardisation des 3 axes ensemble (chacun centré-réduit indépendamment)
X = StandardScaler().fit_transform(feat)
print("Aucun NaN dans X :", not np.isnan(X).any())

Aucun NaN dans X : True


### Cellule 2 — choix du k (silhouette + coude) ET compare KMeans / hiérarchique

In [9]:
rows = []
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
    ag = AgglomerativeClustering(n_clusters=k).fit(X)   # linkage='ward' par défaut
    rows.append({
        'k': k,
        'inertie_kmeans'          : round(km.inertia_, 1),          # pour le coude
        'silhouette_kmeans'       : round(silhouette_score(X, km.labels_), 3),
        'silhouette_hierarchique' : round(silhouette_score(X, ag.labels_), 3),
    })
pd.DataFrame(rows)

,k,inertie_kmeans,silhouette_kmeans,silhouette_hierarchique
0,2,1198.4,0.398,0.342
1,3,851.9,0.400,0.367
2,4,689.2,0.333,0.347
3,5,560.5,0.348,0.294
4,6,499.5,0.307,0.305
5,7,447.1,0.316,0.305


### Cellule 3 — clustering final

In [ ]:
km = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X)
df_final['cluster'] = km.labels_

# Vérif que les 2 algos tombent sur la même structure (robustesse)
from sklearn.metrics import adjusted_rand_score
ag = AgglomerativeClustering(n_clusters=K).fit(X)
print("Accord KMeans vs Hiérarchique (ARI, 1=identique) :",
    round(adjusted_rand_score(km.labels_, ag.labels_), 3))

Accord KMeans vs Hiérarchique (ARI, 1=identique) : 0.699


### Cellule 4 — profil des clusters

In [11]:
profil = df_final.groupby('cluster').agg(
    nb_produits    = ('price', 'size'),
    prix_med       = ('price', 'median'),
    ventes_med     = ('total_sales', 'median'),
    anciennete_med = ('anciennete', 'median'),
    marge_med      = ('tx_marge', 'median'),   # marge en lecture seule (pas dans le clustering)
    ca_total       = ('ca', 'sum'),
).round(1)
profil

,nb_produits,prix_med,ventes_med,anciennete_med,marge_med,ca_total
cluster,,,,,,
0,348,16.2,10.0,825.0,61.2,59954.4
1,188,57.0,4.0,794.0,59.7,51625.6
2,153,24.6,9.0,458.0,61.3,32100.2


Première approche, segmentation sur les axes évidents (prix, ventes) → je retombe sur une intuition métier classique, le ML confirme mais n'apporte rien de neuf. J'enrichis alors avec les dimensions logistiques (rotation, stock immobilisé) → et là émerge un segment que mon analyse manuelle du P6 n'avait pas vu.

# 4 Variables

In [12]:
df_final['stock_mois'].describe()

count    689.000000
mean       2.980996
std        3.468148
min       -0.125000
25%        1.750000
50%        2.400000
75%        3.090909
max       31.250000
Name: stock_mois, dtype: float64

### Étape 1 — anomalie de saisie :

In [13]:
df_var = df_final[df_final['stock_mois'] >= 0].copy()   # stock_mois négatif(s)
print(f"{len(df_final) - len(df_var)} ligne(s) aberrante(s) retirée(s) pour cette variante")

1 ligne(s) aberrante(s) retirée(s) pour cette variante


### Étape 2 — features enrichies (4 axes) :

In [14]:
feat4 = pd.DataFrame(index=df_var.index)
feat4['price_log']      = np.log1p(df_var['price'])
feat4['sales_log']      = np.log1p(df_var['total_sales'])
feat4['anciennete']     = df_var['anciennete']
feat4['stock_mois_log'] = np.log1p(df_var['stock_mois'])

X4 = StandardScaler().fit_transform(feat4)
print("NaN:", np.isnan(X4).any(), "| inf:", np.isinf(X4).any())   # garde-fou

NaN: False | inf: False


### Étape 3 — silhouette de la variante enrichie (à comparer à la table 3 axes) :

In [15]:
rows = []
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X4)
    ag = AgglomerativeClustering(n_clusters=k).fit(X4)
    rows.append({
        'k': k,
        'inertie': round(km.inertia_, 1),
        'silh_kmeans': round(silhouette_score(X4, km.labels_), 3),
        'silh_hierarchique': round(silhouette_score(X4, ag.labels_), 3)}
)
pd.DataFrame(rows)

,k,inertie,silh_kmeans,silh_hierarchique
0,2,1857.7,0.354,0.255
1,3,1420.4,0.387,0.284
2,4,1104.3,0.375,0.315
3,5,943.1,0.353,0.300
4,6,825.3,0.317,0.294
5,7,735.7,0.323,0.233


### Etape 4 — profil des clusters :

In [ ]:
km4 = KMeans(n_clusters=4, random_state=42, n_init=10).fit(X4)
df_var['cluster'] = km4.labels_

df_var.groupby('cluster').agg(
    nb         = ('price', 'size'),
    prix_med   = ('price', 'median'),
    ventes_med = ('total_sales', 'median'),
    anc_med    = ('anciennete', 'median'),
    stock_mois_med = ('stock_mois', 'median'),
    marge_med  = ('tx_marge', 'median'),
).round(1)

,nb,prix_med,ventes_med,anc_med,stock_mois_med,marge_med
cluster,,,,,,
0,326,15.2,10.0,825.0,2.7,61.2
1,142,24.0,9.0,446.5,2.6,61.3
2,188,50.0,4.0,773.0,1.5,61.3
3,32,60.6,6.0,870.5,16.5,40.1


In [17]:
df_var[df_var['cluster'] == 3]['stock_mois'].describe()

count    32.000000
mean     16.930010
std       6.417257
min       6.000000
25%      12.125000
50%      16.500000
75%      21.202381
max      31.250000
Name: stock_mois, dtype: float64

La variante 3 axes maximise la netteté statistique (silhouette 0,40). La variante 4 axes la baisse très légèrement (0,387) mais révèle un segment métier que la baseline masquait : 32 produits à stock dormant et faible marge. Sur le critère valeur métier, la variante enrichie l'emporte malgré une silhouette un poil inférieure — la netteté statistique n'est pas le seul critère de décision.

# VISUALISATION DES CLUSTERS

In [18]:
noms = {0: 'Moteur de CA', 1: 'Nouveautés performantes',
        2: 'Premium', 3: 'Stock dormant'}
df_var['segment'] = df_var['cluster'].map(noms)

fig = px.scatter(
    df_var, x='stock_mois', y='tx_marge', color='segment',
    size='price', hover_data=['total_sales', 'post_title'],
    labels={'stock_mois': 'Couverture de stock (mois)',
            'tx_marge': 'Taux de marge (%)', 'segment': 'Segment'},
    title="Le 4ᵉ segment révélé : stock dormant à faible marge"
)
fig.show()

<div style="background-color: RGB(0,77,64);">
    <h2 style="margin: auto; padding: 20px; color: #fff;">Conclusion — Quatre segments métier exploitables</h2>
</div>

Le clustering retenu (KMeans, k=4, variante enrichie à 4 axes) appliqué au cœur de catalogue
actif (688 produits, hors invendus et hors anomalie de stock) fait émerger **quatre segments
cohérents**, séparés par le **positionnement prix**, l'**ancienneté de la fiche** et la
**rotation de stock** :

| Segment | Volume | Profil (médianes) | Lecture métier |
|---|---|---|---|
| **Moteur de CA** | 326 produits | Prix bas (15 €), fortes ventes (10), ancien, stock fluide (2,7 mois) | Le fond de commerce : entrée de gamme installée qui génère le **plus gros CA total** par effet de volume. |
| **Nouveautés performantes** | 142 produits | Prix moyen (24 €), bonnes ventes (9), **récent (446 j)** | Produits récents qui **démarrent fort** : bonne rotation malgré leur jeunesse. |
| **Premium** | 188 produits | Prix élevé (50 €), faibles ventes (4), ancien | Haut de gamme à **rotation lente assumée** : peu d'unités vendues mais fort panier unitaire. |
| **Stock dormant** | 32 produits | Prix élevé (61 €), **16,5 mois de stock**, **marge faible (40 %)** | **Capital immobilisé** : produits chers, peu rentables, dont le stock met plus d'un an à s'écouler. |

### Recommandations par segment

- **Moteur de CA** → sécuriser la disponibilité (ruptures = perte directe de chiffre), c'est le socle à ne jamais laisser tomber en stock.
- **Nouveautés performantes** → segment à **surveiller et accélérer** : ces produits récents sur-performent, candidats idéaux à la mise en avant (page d'accueil, promo ciblée).
- **Premium** → ne pas juger sur le volume : optimiser le niveau de stock (immobilisation coûteuse) plutôt que pousser les ventes. Rotation lente ≠ problème.
- **Stock dormant** → **action prioritaire** : déstockage ciblé, promo de liquidation ou arrêt de réapprovisionnement. C'est ici que dort du cash sans contrepartie de marge.

### Apport du clustering vs analyse manuelle

Une segmentation intuitive sur le seul couple **prix / ventes** aurait retrouvé les segments les
plus évidents — accessibles à l'œil humain. Mais deux dimensions ajoutées ont révélé des segments
qu'une lecture bivariée masquait : l'**ancienneté** isole les *Nouveautés performantes*, et surtout
la **rotation de stock** fait émerger le segment *Stock dormant* — 32 produits chers à faible marge,
totalement invisibles sur le plan prix/ventes (ils s'y fondaient dans le Premium). La démarche ML
apporte ici une **frontière reproductible, multivariée et scalable**, là où l'analyse manuelle reste
subjective et limitée à deux ou trois axes simultanés.

> **Note méthodologique** — La marge a été **volontairement exclue des variables de clustering**
> (quasi-plate à ~61 % sur la majorité du catalogue, donc non discriminante). Fait notable : le
> segment *Stock dormant* ressort néanmoins avec une marge distincte (40 %) — caractéristique
> obtenue *en lecture seule*, l'axe rotation de stock ayant indirectement capté ce groupe à faible
> rentabilité. Ce que la marge seule n'aurait pas permis de séparer.